<h1>TODO: add a column for the cached path of the images or just use the index of the row as the basis</h1>

In [1]:
import pandas as pd

train_df = pd.read_csv("../dataframes/train_df_final.csv")
test_df = pd.read_csv("../dataframes/test_df_final.csv")
val_df = pd.read_csv("../dataframes/val_df_final.csv")

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 1
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
import sys
import torch

print("Python executable:")
print(sys.executable)

print("\nTorch location:")
print(torch.__file__)

print("\nTorch version:")
print(torch.__version__)

Python executable:
c:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\.venv\Scripts\python.exe

Torch location:
c:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\.venv\Lib\site-packages\torch\__init__.py

Torch version:
2.11.0+cu128


In [4]:
"""Dual-Input Transfer Learning Pipeline for Binary Breast Cancer Classification
(CBIS-DDSM: Benign vs Malignant)

Backbones compared: ResNet50, VGG16, EfficientNet-B2, DenseNet121

Each model receives TWO inputs per sample: the full mammogram and the
cropped lesion. Both images pass through their own instance of the same
backbone architecture, the resulting feature vectors are concatenated,
and a shared fully-connected head produces the final 2-class prediction.

--------------------------------------------------------------------------
CHANGES FROM THE ORIGINAL PIPELINE (to reduce overfitting / improve
generalization on a small dataset with two fully-fine-tuned backbones):

  1. Two-phase training:
       Phase 1 - both backbones FROZEN, only the classifier head trains
                 for HEAD_WARMUP_EPOCHS (lets the head learn to use the
                 pretrained features before any backbone weights move).
       Phase 2 - backbones unfrozen and fine-tuned at a much lower LR
                 than the head (discriminative learning rates).
  2. AdamW instead of Adam (properly decoupled weight decay).
  3. Stronger augmentation: added ColorJitter and a wider rotation range
     on top of the original flip/rotation.
  4. ReduceLROnPlateau patience widened from 3 to 5, since patience=3 on
     a small validation set was reacting to per-epoch noise rather than
     genuine plateaus.
  5. Early-stopping counter resets at the Phase 1 -> Phase 2 transition,
     since unfreezing intentionally perturbs val_loss for a bit and
     shouldn't immediately count against patience.

Everything else (dataset class, metric computation, checkpoint format,
final comparison table) is unchanged from the original script.
--------------------------------------------------------------------------

Prerequisites (must already exist in memory before running this script):
    train_df, val_df, test_df : pandas.DataFrame
        Columns: ["patient_id", "pathology", "image file path",
                  "cropped image file path", "full image cached path",
                  "cropped image cached path"]

Required packages:
    torch, torchvision, pandas, numpy, pillow, scikit-learn

Run as a script (recommended on Windows, because of multiprocessing
DataLoader workers) with train_df / val_df / test_df already loaded.
"""

import os
import time
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------
# Reproducibility
# --------------------------------------------------------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --------------------------------------------------------------------------
# Global configuration
# --------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10
NUM_WORKERS = 4 if os.name != "nt" else 0  # avoid multiprocessing issues on Windows

# --- Two-phase training config ---
HEAD_WARMUP_EPOCHS = 5   # Phase 1 length: backbones frozen, head-only training
HEAD_LR = 1e-4           # LR for the classifier head (both phases)
BACKBONE_LR = 1e-5       # LR for backbones once unfrozen (Phase 2 only)
WEIGHT_DECAY = 1e-4

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

MODEL_NAMES = ["resnet50", "vgg16", "efficientnet_b2", "densenet121"]
DISPLAY_NAMES = {
    "resnet50": "ResNet50",
    "vgg16": "VGG16",
    "efficientnet_b2": "EfficientNet-B2",
    "densenet121": "DenseNet121",
}
CLASS_NAMES = ["Benign", "Malignant"]

FULL_PATH_COL = "full image cached path"
CROP_PATH_COL = "cropped image cached path"
LABEL_COL = "pathology"


# --------------------------------------------------------------------------
# Label encoding
# --------------------------------------------------------------------------
def encode_label(raw_label):
    """
    Encode the 'pathology' column into a binary label.
    Handles both:
      - already-numeric labels (0/1, possibly as int, float, numpy int, or string '0'/'1')
      - text labels like 'BENIGN', 'MALIGNANT', 'BENIGN_WITHOUT_CALLBACK'
    0 = Benign, 1 = Malignant
    """
    try:
        return int(raw_label)
    except (ValueError, TypeError):
        pass
    text = str(raw_label).strip().upper()
    if "MALIGNANT" in text:
        return 1
    return 0


# --------------------------------------------------------------------------
# Transforms
# --------------------------------------------------------------------------
train_transform = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


# --------------------------------------------------------------------------
# Dataset
# --------------------------------------------------------------------------
class DualImageDataset(Dataset):
    """
    Custom dataset returning (full_image, cropped_image, label) for each
    dataframe row. Both images are loaded from the same row, preserving
    the full-mammogram <-> cropped-lesion relationship. Images are cached
    PNGs, already RGB and 224x224, so no resizing is performed here.
    """

    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        full_image = Image.open(row[FULL_PATH_COL]).convert("RGB")
        cropped_image = Image.open(row[CROP_PATH_COL]).convert("RGB")
        full_image = self.transform(full_image)
        cropped_image = self.transform(cropped_image)
        label = encode_label(row[LABEL_COL])
        label = torch.tensor(label, dtype=torch.long)
        return full_image, cropped_image, label


# --------------------------------------------------------------------------
# Backbone factory
# --------------------------------------------------------------------------
def build_backbone(name):
    """
    Builds an ImageNet-pretrained feature extractor (classifier head
    removed, global-average-pooled to a flat feature vector) for the
    requested backbone name. Returns (feature_extractor, feature_dim).
    """
    if name == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V2
        base = models.resnet50(weights=weights)
        modules = list(base.children())[:-1]  # drop fc, keep avgpool
        feature_extractor = nn.Sequential(*modules)
        feature_dim = 2048
    elif name == "vgg16":
        weights = models.VGG16_Weights.IMAGENET1K_V1
        base = models.vgg16(weights=weights)
        feature_extractor = nn.Sequential(
            base.features,
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        feature_dim = 512
    elif name == "efficientnet_b2":
        weights = models.EfficientNet_B2_Weights.IMAGENET1K_V1
        base = models.efficientnet_b2(weights=weights)
        feature_extractor = nn.Sequential(
            base.features,
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        feature_dim = 1408
    elif name == "densenet121":
        weights = models.DenseNet121_Weights.IMAGENET1K_V1
        base = models.densenet121(weights=weights)
        feature_extractor = nn.Sequential(
            base.features,
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        feature_dim = 1024
    else:
        raise ValueError(f"Unknown backbone name: {name}")
    return feature_extractor, feature_dim


# --------------------------------------------------------------------------
# Dual-input model
# --------------------------------------------------------------------------
class DualInputModel(nn.Module):
    """
    Generic dual-input architecture:
        full image    -> feature extractor A -\\
                                                 >-- concat -> FC head -> logits
        cropped image -> feature extractor B -/

    Both feature extractors use the same backbone architecture (specified
    by `backbone_name`) but are separate instances (independent weights).
    """

    def __init__(self, backbone_name, num_classes=2, dropout=0.5):
        super().__init__()
        self.backbone_name = backbone_name
        self.full_extractor, feat_dim_full = build_backbone(backbone_name)
        self.crop_extractor, feat_dim_crop = build_backbone(backbone_name)
        assert feat_dim_full == feat_dim_crop
        combined_dim = feat_dim_full + feat_dim_crop
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, full_image, cropped_image):
        full_feat = self.full_extractor(full_image)
        full_feat = torch.flatten(full_feat, 1)
        crop_feat = self.crop_extractor(cropped_image)
        crop_feat = torch.flatten(crop_feat, 1)
        combined = torch.cat([full_feat, crop_feat], dim=1)
        logits = self.classifier(combined)
        return logits

    def set_backbones_trainable(self, trainable: bool):
        for extractor in (self.full_extractor, self.crop_extractor):
            for param in extractor.parameters():
                param.requires_grad = trainable
            extractor.train(trainable)   # eval() -> freezes BN running stats too


# --------------------------------------------------------------------------
# Metric computation helper
# --------------------------------------------------------------------------
def compute_metrics(labels, preds, probs):
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float("nan")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": auc}


# --------------------------------------------------------------------------
# Shared train / eval epoch runner
# --------------------------------------------------------------------------
def run_epoch(model, loader, device, criterion, optimizer=None, scaler=None, use_amp=False):
    """
    Runs one pass over `loader`. If `optimizer` is given, does a training
    epoch (gradients + weight updates); otherwise runs in eval mode only.
    Returns (avg_loss, metrics_dict).
    """
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []

    grad_context = torch.enable_grad() if is_train else torch.no_grad()
    with grad_context:
        for full_imgs, crop_imgs, labels in loader:
            full_imgs = full_imgs.to(device, non_blocking=True)
            crop_imgs = crop_imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if is_train:
                optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(full_imgs, crop_imgs)
                loss = criterion(outputs, labels)

            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            running_loss += loss.item() * labels.size(0)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(all_labels, all_preds, all_probs)
    return avg_loss, metrics


# --------------------------------------------------------------------------
# Training loop (one model, two phases)
# --------------------------------------------------------------------------
def train_one_model(model_name, train_loader, val_loader, device):
    print(f"\n{'=' * 80}")
    print(f"Training model: {DISPLAY_NAMES[model_name]}")
    print(f"{'=' * 80}")

    model = DualInputModel(model_name).to(device)
    criterion = nn.CrossEntropyLoss()
    use_amp = device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    checkpoint_path = f"best_{model_name}.pth"
    best_val_loss = float("inf")
    best_val_acc = 0.0
    global_epoch = 0

    # ---------------- Phase 1: head-only warmup, backbones frozen ----------------
    model.set_backbones_trainable(False)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=HEAD_LR,
        weight_decay=WEIGHT_DECAY,
    )

    print(f"-- Phase 1: head warmup ({HEAD_WARMUP_EPOCHS} epochs, backbones frozen) --")
    for _ in range(HEAD_WARMUP_EPOCHS):
        epoch_start = time.time()
        train_loss, train_metrics = run_epoch(
            model, train_loader, device, criterion, optimizer, scaler, use_amp
        )
        val_loss, val_metrics = run_epoch(model, val_loader, device, criterion)
        epoch_time = time.time() - epoch_start
        global_epoch += 1

        print(
            f"Epoch {global_epoch:3d}/{NUM_EPOCHS} [warmup] | "
            f"train_loss: {train_loss:.4f} | train_acc: {train_metrics['accuracy']:.4f} | "
            f"val_loss: {val_loss:.4f} | val_acc: {val_metrics['accuracy']:.4f} | "
            f"val_prec: {val_metrics['precision']:.4f} | val_rec: {val_metrics['recall']:.4f} | "
            f"val_f1: {val_metrics['f1']:.4f} | val_auc: {val_metrics['roc_auc']:.4f} | "
            f"lr: {optimizer.param_groups[0]['lr']:.2e} | time: {epoch_time:.1f}s"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_metrics["accuracy"]
            torch.save(
                {
                    "epoch": global_epoch - 1,
                    "model_state_dict": model.state_dict(),
                    "val_accuracy": val_metrics["accuracy"],
                    "val_loss": val_loss,
                },
                checkpoint_path,
            )

    # ---------------- Phase 2: unfreeze backbones, discriminative LR ----------------
    model.set_backbones_trainable(True)
    optimizer = optim.AdamW(
        [
            {"params": model.full_extractor.parameters(), "lr": BACKBONE_LR},
            {"params": model.crop_extractor.parameters(), "lr": BACKBONE_LR},
            {"params": model.classifier.parameters(), "lr": HEAD_LR},
        ],
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.1, patience=5
    )

    # Reset patience: unfreezing intentionally perturbs val_loss for a bit
    # and shouldn't immediately count against early stopping.
    epochs_no_improve = 0

    print("-- Phase 2: full fine-tuning (backbones unfrozen, discriminative LR) --")
    remaining_epochs = max(NUM_EPOCHS - HEAD_WARMUP_EPOCHS, 0)
    for _ in range(remaining_epochs):
        epoch_start = time.time()
        train_loss, train_metrics = run_epoch(
            model, train_loader, device, criterion, optimizer, scaler, use_amp
        )
        val_loss, val_metrics = run_epoch(model, val_loader, device, criterion)
        scheduler.step(val_loss)
        epoch_time = time.time() - epoch_start
        global_epoch += 1

        current_lrs = "/".join(f"{g['lr']:.1e}" for g in optimizer.param_groups)
        print(
            f"Epoch {global_epoch:3d}/{NUM_EPOCHS} | "
            f"train_loss: {train_loss:.4f} | train_acc: {train_metrics['accuracy']:.4f} | "
            f"val_loss: {val_loss:.4f} | val_acc: {val_metrics['accuracy']:.4f} | "
            f"val_prec: {val_metrics['precision']:.4f} | val_rec: {val_metrics['recall']:.4f} | "
            f"val_f1: {val_metrics['f1']:.4f} | val_auc: {val_metrics['roc_auc']:.4f} | "
            f"lr(bb/bb/head): {current_lrs} | time: {epoch_time:.1f}s"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_metrics["accuracy"]
            epochs_no_improve = 0
            torch.save(
                {
                    "epoch": global_epoch - 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "val_accuracy": val_metrics["accuracy"],
                    "val_loss": val_loss,
                },
                checkpoint_path,
            )
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                print(
                    f"Early stopping triggered at epoch {global_epoch} "
                    f"(no improvement for {EARLY_STOPPING_PATIENCE} epochs)."
                )
                break

    # Reload best checkpoint before returning/testing
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(
        f"Best checkpoint for {DISPLAY_NAMES[model_name]}: "
        f"epoch {checkpoint['epoch'] + 1}, "
        f"val_loss={checkpoint['val_loss']:.4f}, "
        f"val_accuracy={checkpoint['val_accuracy']:.4f}"
    )
    return model, checkpoint


# --------------------------------------------------------------------------
# Test-set evaluation
# --------------------------------------------------------------------------
def evaluate_model(model, test_loader, device, model_name):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for full_imgs, crop_imgs, labels in test_loader:
            full_imgs = full_imgs.to(device, non_blocking=True)
            crop_imgs = crop_imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(full_imgs, crop_imgs)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    metrics = compute_metrics(all_labels, all_preds, all_probs)
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, zero_division=0)

    print(f"\n--- Test results: {DISPLAY_NAMES[model_name]} ---")
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall   : {metrics['recall']:.4f}")
    print(f"F1-score : {metrics['f1']:.4f}")
    print(f"ROC-AUC  : {metrics['roc_auc']:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print("Classification Report:")
    print(report)

    metrics["confusion_matrix"] = cm
    metrics["classification_report"] = report
    return metrics


# --------------------------------------------------------------------------
# Main pipeline
# --------------------------------------------------------------------------
def run_pipeline(train_df, val_df, test_df):
    print(f"Using device: {DEVICE}")

    train_dataset = DualImageDataset(train_df, train_transform)
    val_dataset = DualImageDataset(val_df, eval_transform)
    test_dataset = DualImageDataset(test_df, eval_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    all_test_results = {}
    for model_name in MODEL_NAMES:
        trained_model, _ = train_one_model(model_name, train_loader, val_loader, DEVICE)
        test_metrics = evaluate_model(trained_model, test_loader, DEVICE, model_name)
        all_test_results[model_name] = test_metrics
        del trained_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"\n{'=' * 88}")
    print("FINAL MODEL COMPARISON (Test Set)")
    print(f"{'=' * 88}")
    header = f"{'Model':<20}{'Accuracy':<12}{'Precision':<13}{'Recall':<10}{'F1-score':<12}{'ROC-AUC':<10}"
    print(header)
    print("-" * len(header))
    for model_name in MODEL_NAMES:
        m = all_test_results[model_name]
        print(
            f"{DISPLAY_NAMES[model_name]:<20}"
            f"{m['accuracy']:<12.4f}"
            f"{m['precision']:<13.4f}"
            f"{m['recall']:<10.4f}"
            f"{m['f1']:<12.4f}"
            f"{m['roc_auc']:<10.4f}"
        )

    best_model_name = max(all_test_results, key=lambda k: all_test_results[k]["roc_auc"])
    print(
        f"\nBest-performing model (by test ROC-AUC): "
        f"{DISPLAY_NAMES[best_model_name]} "
        f"(ROC-AUC = {all_test_results[best_model_name]['roc_auc']:.4f})"
    )
    return all_test_results

In [5]:
print(train_df["pathology"].value_counts())
print(val_df["pathology"].value_counts())
print(test_df["pathology"].value_counts())

pathology
0    1536
1    1063
Name: count, dtype: int64
pathology
0    195
1    133
Name: count, dtype: int64
pathology
0    188
1    138
Name: count, dtype: int64


In [6]:
labels_check = [encode_label(l) for l in val_df["pathology"]]
print(np.bincount(labels_check))  # should now print [195 133]

[195 133]


In [7]:
# --------------------------------------------------------------------------
# Entry point
# --------------------------------------------------------------------------
if __name__ == "__main__":
    # train_df, val_df, and test_df must already be defined in this session
    # (e.g. loaded earlier in the same script/notebook) before running this.
    results = run_pipeline(train_df, val_df, test_df)

Using device: cuda

Training model: ResNet50
-- Phase 1: head warmup (5 epochs, backbones frozen) --
Epoch   1/100 [warmup] | train_loss: 0.6682 | train_acc: 0.5968 | val_loss: 0.6364 | val_acc: 0.6006 | val_prec: 0.6000 | val_rec: 0.0451 | val_f1: 0.0839 | val_auc: 0.7623 | lr: 1.00e-04 | time: 71.8s
Epoch   2/100 [warmup] | train_loss: 0.6305 | train_acc: 0.6383 | val_loss: 0.5987 | val_acc: 0.6799 | val_prec: 0.8043 | val_rec: 0.2782 | val_f1: 0.4134 | val_auc: 0.7671 | lr: 1.00e-04 | time: 67.5s
Epoch   3/100 [warmup] | train_loss: 0.6140 | train_acc: 0.6629 | val_loss: 0.5813 | val_acc: 0.7073 | val_prec: 0.7846 | val_rec: 0.3835 | val_f1: 0.5152 | val_auc: 0.7683 | lr: 1.00e-04 | time: 49.9s
Epoch   4/100 [warmup] | train_loss: 0.6011 | train_acc: 0.6653 | val_loss: 0.5596 | val_acc: 0.7226 | val_prec: 0.7143 | val_rec: 0.5263 | val_f1: 0.6061 | val_auc: 0.7880 | lr: 1.00e-04 | time: 48.6s
Epoch   5/100 [warmup] | train_loss: 0.5814 | train_acc: 0.6903 | val_loss: 0.5486 | val_ac